# Reasoning SLM with TRL and a PyTorch training loop

This notebook shows two ways to improve a small language model on simple reasoning tasks:

1. TRL + GRPOTrainer with a verifiable reward
2. A custom `nn.Module` + PyTorch training loop

We compare the model before and after training on the same prompts to show the behavioral shift. The goal is to keep the setup compact and readable while making the training mechanics explicit.

## 1. Setup

Install the required libraries if they are not already available.

In [ ]:
!pip install -q transformers trl datasets accelerate peft sentencepiece torch

In [ ]:


import os
import re
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from trl import GRPOConfig, GRPOTrainer

torch.manual_seed(0)
np.random.seed(0)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

## 2. Baseline model: before-training inference

We pick a small base model and test it on a few arithmetic-style prompts before we train it.

In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-135M-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
).to(DEVICE).eval()

PROMPTS = [
    "What is 23 + 17? Reply with only the number.",
    "If a box has 5 apples and 3 are removed, how many remain? Reply with only the number.",
    "What is 12 * 8? Reply with only the number.",
    "Tom has 2 books. He buys 3 more. How many books does he have? Reply with only the number.",
]

def generate_text(model, prompts, max_new_tokens=32, temperature=0.8):
    enc = tokenizer(prompts, return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_k=0,
            pad_token_id=tokenizer.pad_token_id,
        )
    return tokenizer.batch_decode(out[:, enc["input_ids"].shape[1]:], skip_special_tokens=True)

print("=== BEFORE TRAINING ===")
before_outputs = generate_text(base_model, PROMPTS)
for i, prompt in enumerate(PROMPTS, 1):
    print(f"\n--- before training prompt {i} ---")
    print(prompt)
    print(before_outputs[i - 1])

## 3. Train with TRL GRPOTrainer

We use a simple verifier: reward is 1.0 if the final number in the completion matches the correct answer; otherwise 0.0. This creates a clean reward signal for structured reasoning.

In [ ]:
math_tasks = [
    ("What is 23 + 17? Reply with only the number.", "40"),
    ("If a box has 5 apples and 3 are removed, how many remain? Reply with only the number.", "2"),
    ("What is 12 * 8? Reply with only the number.", "96"),
    ("Tom has 2 books. He buys 3 more. How many books does he have? Reply with only the number.", "5"),
    ("What is 7 + 6? Reply with only the number.", "13"),
    ("A toy costs 9 dollars and a sticker costs 4 dollars. Total cost? Reply with only the number.", "13"),
    ("What is 5 * 9? Reply with only the number.", "45"),
    ("A shelf has 11 books and 4 are added. How many now? Reply with only the number.", "15"),
]

math_ds = Dataset.from_list([
    {"prompt": [{"role": "user", "content": q}], "answer": a}
    for q, a in math_tasks
])

def correctness_reward(completions, answer, **kwargs):
    rewards = []
    for completion, ans in zip(completions, answer):
        text = completion[0]["content"]
        numbers = re.findall(r"-?\d+", text)
        reward = 1.0 if numbers and numbers[-1] == ans else 0.0
        rewards.append(reward)
    return rewards

grpo_args = GRPOConfig(
    output_dir="/tmp/slm_reasoning_grpo",
    num_generations=4,
    max_completion_length=24,
    max_steps=6,
    per_device_train_batch_size=2,
    learning_rate=5e-6,
    beta=0.04,
    logging_steps=1,
    temperature=0.9,
    report_to="none",
)

trainer = GRPOTrainer(
    model=base_model,
    reward_funcs=correctness_reward,
    args=grpo_args,
    train_dataset=math_ds,
    processing_class=tokenizer,
)

trainer.train()
trained_model = trainer.model
trained_model.eval()
print("GRPO training finished.")

## 4. After-training inference: GRPO version

Now rerun the same prompts and compare the outputs.

In [ ]:
print("=== AFTER TRAINING (GRPO) ===")
after_grpo_outputs = generate_text(trained_model, PROMPTS)
for i, prompt in enumerate(PROMPTS, 1):
    print(f"\n--- after GRPO prompt {i} ---")
    print(prompt)
    print(after_grpo_outputs[i - 1])

## 5. Custom PyTorch training loop

This second version shows the same idea without `trl`. A small reasoning model is trained directly with a manual loop.

In [ ]:
class TinyReasoner(nn.Module):
    def __init__(self, vocab_size=64, embed_dim=16, hidden_dim=32):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.net = nn.Sequential(
            nn.Linear(embed_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
        )
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, tokens):
        x = self.embedding(tokens)
        x = x.mean(dim=1)
        x = self.net(x)
        return self.head(x)

def encode_expr(expr):
    char_to_id = {ch: i + 1 for i, ch in enumerate("0123456789+-*= ")}
    return torch.tensor([char_to_id[ch] for ch in expr], dtype=torch.long)

def encode_answer(ans):
    char_to_id = {ch: i + 1 for i, ch in enumerate("0123456789")}
    return torch.tensor([char_to_id[ans]], dtype=torch.long)

training_pairs = [
    ("3 + 5", "8"),
    ("7 + 2", "9"),
    ("9 - 4", "5"),
    ("6 * 3", "18"),
    ("8 + 1", "9"),
    ("10 - 3", "7"),
    ("4 * 5", "20"),
    ("11 + 2", "13"),
]

tiny_model = TinyReasoner(vocab_size=64, embed_dim=16, hidden_dim=32)
optimizer = torch.optim.AdamW(tiny_model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

for epoch in range(250):
    total_loss = 0.0
    for expr, ans in training_pairs:
        x = encode_expr(expr).unsqueeze(0)
        y = encode_answer(ans)
        logits = tiny_model(x)
        loss = criterion(logits, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    if epoch % 50 == 0:
        print(f"epoch {epoch}: loss = {total_loss / len(training_pairs):.4f}")

print("Custom PyTorch reasoning loop finished.")

## 6. Before-vs-after check for the custom PyTorch model

This is a smaller sanity check showing how a tiny model can become more consistent after training.

In [ ]:
def predict_tiny_reasoner(model, expr):
    x = encode_expr(expr).unsqueeze(0)
    with torch.no_grad():
        logits = model(x)
        pred = torch.argmax(logits, dim=-1)[0].item()
    return str(pred)

print("=== CUSTOM PYTORCH MODEL CHECK ===")
for expr, ans in training_pairs:
    print(f"{expr} -> predicted {predict_tiny_reasoner(tiny_model, expr)} | target {ans}")

## 7. Summary

This notebook demonstrates the core pattern behind reasoning tuning for a small model:

- start with a non-reasoning base model
- evaluate it before training
- apply a rule-based verifier or reward function
- train with either `trl.GRPOTrainer` or a custom PyTorch loop
- compare the outputs before and after training

The same workflow scales to larger base models and more realistic training datasets.